# Parte Específica: EDA distribucional do BankChurners

Dataset tratado em `eda_unknown.ipynb`: `out/parte_especifica/bankchurners_tratado.csv`
("Unknown" mantido como categoria própria).

Pilar distribucional: como cada variável se distribui, sozinha.

In [1]:
# [IA] Claude Code – importacao das bibliotecas, leitura do dataset tratado e pasta de saida
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Sem isso o matplotlib le o "$" das faixas de renda ("$40K - $60K") como formula matematica
plt.rcParams["text.parse_math"] = False

CAMINHO_DADOS = Path("../../out/parte_especifica/bankchurners_tratado.csv")
PASTA_SAIDA = Path("../../out/parte_especifica")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CAMINHO_DADOS)
COLUNA_ALVO = "Attrition_Flag"
print(df.shape)
df.head()

(10127, 21)


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000


In [2]:
# [IA] Claude Code – grupos de variaveis (CLIENTNUM e identificador, fica fora da analise)
colunas_numericas = df.select_dtypes(include="number").columns.drop("CLIENTNUM").tolist()
colunas_categoricas = df.select_dtypes(exclude="number").columns.drop(COLUNA_ALVO).tolist()

# Ordem natural das categorias ordinais (o value_counts ordenaria por frequencia)
ORDEM_CATEGORIAS = {
    "Education_Level": ["Uneducated", "High School", "College", "Graduate", "Post-Graduate", "Doctorate", "Unknown"],
    "Income_Category": ["Less than $40K", "$40K - $60K", "$60K - $80K", "$80K - $120K", "$120K +", "Unknown"],
    "Card_Category": ["Blue", "Silver", "Gold", "Platinum"],
}
print("Numericas:", colunas_numericas)
print("Categoricas:", colunas_categoricas)

Numericas: ['Customer_Age', 'Dependent_count', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']
Categoricas: ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']


## 1. Variável alvo

In [3]:
# [IA] Claude Code – contagem e proporcao da variavel alvo
alvo = pd.DataFrame({
    "qtd": df[COLUNA_ALVO].value_counts(),
    "pct": (df[COLUNA_ALVO].value_counts(normalize=True) * 100).round(2),
})
alvo

,qtd,pct
Attrition_Flag,,
Existing Customer,8500,83.93
Attrited Customer,1627,16.07


In [4]:
# [IA] Claude Code – grafico de barras da variavel alvo
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(alvo.index, alvo["qtd"], color=["#4C72B0", "#C44E52"])
for x, (q, p) in enumerate(zip(alvo["qtd"], alvo["pct"])):
    ax.text(x, q, f"{q} ({p:.1f}%)", ha="center", va="bottom")
ax.set_title("BankChurners: clientes ativos x clientes que cancelaram")
ax.set_xlabel("Situacao do cliente (Attrition_Flag)")
ax.set_ylabel("Quantidade de clientes")
ax.set_ylim(0, alvo["qtd"].max() * 1.15)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_distribuicao_alvo.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\luisgx\AppData\Local\Temp\ipykernel_35704\38506557.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Variáveis numéricas

In [5]:
# [IA] Claude Code – medidas resumo + assimetria e curtose de cada variavel numerica
resumo = df[colunas_numericas].describe().T
resumo["assimetria"] = df[colunas_numericas].skew()
resumo["curtose"] = df[colunas_numericas].kurt()
resumo["cv_pct"] = (resumo["std"] / resumo["mean"] * 100)  # coeficiente de variacao
resumo.round(3)

,count,mean,std,min,25%,50%,75%,max,assimetria,curtose,cv_pct
Customer_Age,10127.0,46.326,8.017,26.0,41.000,46.000,52.000,73.000,-0.034,-0.289,17.305
Dependent_count,10127.0,2.346,1.299,0.0,1.000,2.000,3.000,5.000,-0.021,-0.683,55.362
Months_on_book,10127.0,35.928,7.986,13.0,31.000,36.000,40.000,56.000,-0.107,0.400,22.229
Total_Relationship_Count,10127.0,3.813,1.554,1.0,3.000,4.000,5.000,6.000,-0.162,-1.006,40.770
Months_Inactive_12_mon,10127.0,2.341,1.011,0.0,2.000,2.000,3.000,6.000,0.633,1.099,43.167
Contacts_Count_12_mon,10127.0,2.455,1.106,0.0,2.000,2.000,3.000,6.000,0.011,0.001,45.054
Credit_Limit,10127.0,8631.954,9088.777,1438.3,2555.000,4549.000,11067.500,34516.000,1.667,1.809,105.292
Total_Revolving_Bal,10127.0,1162.814,814.987,0.0,359.000,1276.000,1784.000,2517.000,-0.149,-1.146,70.088
Avg_Open_To_Buy,10127.0,7469.140,9090.685,3.0,1324.500,3474.000,9859.000,34516.000,1.662,1.799,121.710
Total_Amt_Chng_Q4_Q1,10127.0,0.760,0.219,0.0,0.631,0.736,0.859,3.397,1.732,9.994,28.845


In [6]:
# [IA] Claude Code – classificar a forma pela assimetria (|a| < 0.5 ~ simetrica; 0.5 a 1 moderada; > 1 forte)
def forma(a):
    if abs(a) < 0.5:
        return "aproximadamente simetrica"
    lado = "a direita" if a > 0 else "a esquerda"
    return f"assimetria {'moderada' if abs(a) <= 1 else 'forte'} {lado}"

pd.DataFrame({
    "assimetria": resumo["assimetria"].round(2),
    "forma": resumo["assimetria"].map(forma),
    "qtd_valores_distintos": df[colunas_numericas].nunique(),
}).sort_values("assimetria")

,assimetria,forma,qtd_valores_distintos
Total_Relationship_Count,-0.16,aproximadamente simetrica,6
Total_Revolving_Bal,-0.15,aproximadamente simetrica,1974
Months_on_book,-0.11,aproximadamente simetrica,44
Customer_Age,-0.03,aproximadamente simetrica,45
Dependent_count,-0.02,aproximadamente simetrica,6
Contacts_Count_12_mon,0.01,aproximadamente simetrica,7
Total_Trans_Ct,0.15,aproximadamente simetrica,126
Months_Inactive_12_mon,0.63,assimetria moderada a direita,7
Avg_Utilization_Ratio,0.72,assimetria moderada a direita,964
Avg_Open_To_Buy,1.66,assimetria forte a direita,6813


In [7]:
# [IA] Claude Code – histogramas de todas as variaveis numericas (com media e mediana)
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for ax, coluna in zip(axes.flat, colunas_numericas):
    discreta = df[coluna].nunique() <= 15
    sns.histplot(df[coluna], bins=30, discrete=discreta, kde=not discreta, ax=ax, color="#4C72B0")
    ax.axvline(df[coluna].mean(), color="#C44E52", linestyle="--", linewidth=1.5, label="media")
    ax.axvline(df[coluna].median(), color="#55A868", linestyle="-", linewidth=1.5, label="mediana")
    ax.set_title(coluna)
    ax.set_xlabel(coluna)
    ax.set_ylabel("Quantidade de clientes")
    ax.legend(fontsize=8)
for ax in axes.flat[len(colunas_numericas):]:
    ax.set_visible(False)
fig.suptitle("BankChurners: distribuicao das variaveis numericas", fontsize=16)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_histogramas_numericas.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\luisgx\AppData\Local\Temp\ipykernel_35704\908774717.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# [IA] Claude Code – variaveis com assimetria forte vistas em escala log (so para leitura da forma)
fortes = resumo.index[resumo["assimetria"].abs() > 1].tolist()
fig, axes = plt.subplots(1, len(fortes), figsize=(5 * len(fortes), 4), squeeze=False)
for ax, coluna in zip(axes.flat, fortes):
    sns.histplot(np.log1p(df[coluna]), bins=30, kde=True, ax=ax, color="#8172B3")
    ax.set_title(f"log(1 + {coluna})")
    ax.set_xlabel(f"log(1 + {coluna})")
    ax.set_ylabel("Quantidade de clientes")
fig.suptitle("Variaveis com assimetria forte em escala logaritmica", fontsize=14)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_histogramas_log.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\luisgx\AppData\Local\Temp\ipykernel_35704\1354998493.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# [IA] Claude Code – concentracoes em um unico valor (picos que aparecem nos histogramas)
picos = pd.DataFrame({
    "valor_mais_frequente": df[colunas_numericas].mode().iloc[0],
    "pct_no_valor_mais_frequente": [
        (df[c] == df[c].mode().iloc[0]).mean() * 100 for c in colunas_numericas
    ],
}).round(2).sort_values("pct_no_valor_mais_frequente", ascending=False)
picos.head(8)

,valor_mais_frequente,pct_no_valor_mais_frequente
Months_Inactive_12_mon,3.0,37.98
Contacts_Count_12_mon,3.0,33.38
Dependent_count,3.0,26.98
Total_Revolving_Bal,0.0,24.39
Avg_Utilization_Ratio,0.0,24.39
Months_on_book,36.0,24.32
Total_Relationship_Count,3.0,22.76
Credit_Limit,34516.0,5.02


In [10]:
# [IA] Claude Code – Total_Trans_Amt parece ter mais de um "morro"; ver o histograma com mais detalhe
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df["Total_Trans_Amt"], bins=80, ax=ax, color="#4C72B0")
ax.set_title("Total_Trans_Amt: valor total transacionado em 12 meses (80 faixas)")
ax.set_xlabel("Total_Trans_Amt (US$)")
ax.set_ylabel("Quantidade de clientes")
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_hist_total_trans_amt.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\luisgx\AppData\Local\Temp\ipykernel_35704\3241818963.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Leitura das distribuições numéricas

Pelo critério da tabela `forma` (|assimetria| < 0,5), são aproximadamente simétricas `Customer_Age`, `Dependent_count`, `Months_on_book`, `Total_Relationship_Count`, `Total_Revolving_Bal`, `Contacts_Count_12_mon` e `Total_Trans_Ct`. `Months_Inactive_12_mon` e `Avg_Utilization_Ratio` têm assimetria moderada à direita. `Credit_Limit`, `Avg_Open_To_Buy`, `Total_Amt_Chng_Q4_Q1`, `Total_Trans_Amt` e `Total_Ct_Chng_Q4_Q1` têm assimetria forte à direita. `Total_Revolving_Bal` parece simétrica pelo valor de assimetria (-0,149) porque essa medida resume a distribuição em um único número; no gráfico, porém, há uma concentração muito grande em zero e outro agrupamento nos valores positivos, caracterizando uma distribuição bimodal. As distribuições com múltiplos picos aparecem com maior clareza em `Total_Revolving_Bal`, `Total_Trans_Amt` e `Total_Trans_Ct`; `Credit_Limit` e `Avg_Open_To_Buy` também mostram uma concentração secundária próxima do valor máximo observado.

## 3. Variáveis categóricas

In [11]:
# [IA] Claude Code – frequencias absolutas e relativas de cada variavel categorica
for coluna in colunas_categoricas:
    ordem = ORDEM_CATEGORIAS.get(coluna, df[coluna].value_counts().index.tolist())
    tabela = pd.DataFrame({
        "qtd": df[coluna].value_counts().reindex(ordem),
        "pct": (df[coluna].value_counts(normalize=True) * 100).reindex(ordem).round(2),
    })
    print(f"\n{coluna}")
    print(tabela.to_string())


Gender
         qtd    pct
Gender             
F       5358  52.91
M       4769  47.09

Education_Level
                  qtd    pct
Education_Level             
Uneducated       1487  14.68
High School      2013  19.88
College          1013  10.00
Graduate         3128  30.89
Post-Graduate     516   5.10
Doctorate         451   4.45
Unknown          1519  15.00

Marital_Status
                 qtd    pct
Marital_Status             
Married         4687  46.28
Single          3943  38.94
Unknown          749   7.40
Divorced         748   7.39

Income_Category
                  qtd    pct
Income_Category             
Less than $40K   3561  35.16
$40K - $60K      1790  17.68
$60K - $80K      1402  13.84
$80K - $120K     1535  15.16
$120K +           727   7.18
Unknown          1112  10.98

Card_Category
                qtd    pct
Card_Category             
Blue           9436  93.18
Silver          555   5.48
Gold            116   1.15
Platinum         20   0.20


In [12]:
# [IA] Claude Code – barras horizontais das categoricas (ordinais na ordem natural; Unknown em cinza)
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, coluna in zip(axes.flat, colunas_categoricas):
    ordem = ORDEM_CATEGORIAS.get(coluna, df[coluna].value_counts().sort_values().index.tolist())
    contagem = df[coluna].value_counts().reindex(ordem)
    if coluna in ORDEM_CATEGORIAS:
        contagem = contagem.iloc[::-1]  # primeira categoria em cima
    cores = ["#B0B0B0" if c == "Unknown" else "#4C72B0" for c in contagem.index]
    ax.barh(contagem.index.astype(str), contagem.values, color=cores)
    for y, v in enumerate(contagem.values):
        ax.text(v, y, f" {v / len(df) * 100:.1f}%", va="center", fontsize=9)
    ax.set_title(coluna)
    ax.set_xlabel("Quantidade de clientes")
    ax.set_ylabel(coluna)
    ax.set_xlim(0, contagem.max() * 1.2)
for ax in axes.flat[len(colunas_categoricas):]:
    ax.set_visible(False)
fig.suptitle("BankChurners: distribuicao das variaveis categoricas (cinza = Unknown)", fontsize=15)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_barras_categoricas.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\luisgx\AppData\Local\Temp\ipykernel_35704\4069500762.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Leitura das distribuições categóricas

As categorias dominantes são `F` em `Gender` (52,91%), `Graduate` em `Education_Level` (30,89%), `Married` em `Marital_Status` (46,28%), `Less than $40K` em `Income_Category` (35,16%) e `Blue` em `Card_Category` (93,18%). Há `Unknown` em `Education_Level` (15,00%, 1.519 clientes), `Marital_Status` (7,40%, 749 clientes) e `Income_Category` (10,98%, 1.112 clientes). `Gender` e `Card_Category` não possuem `Unknown`.

## Observações do pilar distribucional

Chamam atenção as distribuições bimodais ou multimodais de `Total_Revolving_Bal`, `Total_Trans_Amt` e `Total_Trans_Ct`, além da forte assimetria à direita das variáveis de limite e de variação transacional. A concentração de `Total_Revolving_Bal` em zero merece investigação, pois pode separar clientes que não carregam saldo dos demais. Nas categorias, `Blue` domina `Card_Category`, enquanto `Unknown` é relevante em `Education_Level`, `Income_Category` e `Marital_Status`. Vale cruzar `Total_Trans_Ct`, `Total_Trans_Amt`, `Total_Revolving_Bal` e `Avg_Utilization_Ratio` com `Attrition_Flag` para verificar se esses perfis de uso e concentração de saldo diferenciam clientes que ficaram dos que cancelaram.